# Notebook — Motor de matching y evaluación (Levenshtein, Jaro-Winkler, Fonético, Embeddings)

Este notebook es un **laboratorio de evaluación** para escoger un modelo de matching de nombres.

**Qué hace (alto nivel):**
1. Carga **terceros sintéticos** (CSV) con la etiqueta `coincidencia` (ground truth).
2. Carga **consolidado** (SQLite) como base de búsqueda.
3. Normaliza texto para que las comparaciones sean consistentes.
4. Genera candidatos (blocking) para acelerar el matching.
5. Calcula scores para 4 enfoques (Levenshtein, Jaro-Winkler, Fonético, Embeddings).
6. Evalúa con **Precision / Recall / F1** en varios umbrales.
7. Permite inspeccionar errores (FP/FN) para mejorar y decidir.

> Tip: Primero corre con `df_sample` (muestra) para iterar rápido. Cuando estés listo, aumenta el tamaño o corre todo.


In [ ]:
# ============================================
# 1) IMPORTS
# --------------------------------------------
# Este notebook compara varios "modelos" de matching de nombres:
# - Levenshtein
# - Jaro-Winkler
# - Fonético (Double Metaphone)
# - Embeddings (sentence-transformers + cosine similarity)
#
# Además:
# - Carga datos desde CSV (terceros sintéticos) y SQLite (consolidado)
# - Normaliza texto para que la comparación sea consistente
# - Genera candidatos (blocking) para acelerar el matching
# - Evalúa con Precision / Recall / F1 a distintos umbrales
# ============================================

import os               # Variables de entorno (por si DB_PATH existe)
import json
import re               # Normalización de texto con regex
import sqlite3          # Conexión a SQLite
from pathlib import Path

import numpy as np      # Arrays y operaciones numéricas
import pandas as pd     # DataFrames
import matplotlib.pyplot as plt # Graficos

# RapidFuzz: implementaciones rápidas de distancias/similitudes
from rapidfuzz.distance import Levenshtein, JaroWinkler

# Metaphone: algoritmo fonético (útil para errores por pronunciación/ortografía)
from metaphone import doublemetaphone

# Embeddings: representación semántica del nombre
from sentence_transformers import SentenceTransformer

# Métricas de clasificación (score -> predicción por umbral)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import precision_recall_curve, roc_curve, auc

# Similaridad coseno para embeddings
from sklearn.metrics.pairwise import cosine_similarity



c:\Users\pc\OneDrive\Documentos\Data project\tusdatos\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuración de rutas
Ajusta estas rutas si tu proyecto usa otra estructura.


In [2]:
# ============================================
# 2) CONFIGURACIÓN DE RUTAS Y FUENTES
# --------------------------------------------
# - DB_FILE: base SQLite donde está la tabla "consolidado"
# - TERCEROS_CSV: CSV generado con tu script (terceros_sinteticos.csv)
#
# Nota:
# Path(".").resolve().parents[1] asume que el notebook está en una carpeta
# tipo: <repo>/notebooks/<este_notebook>.ipynb
# Si mueves el notebook, ajusta parents[...] o usa una ruta absoluta.
# ============================================

PROJECT_ROOT = Path(".").resolve().parents[1]     # Raíz del repo (ajusta si cambia tu estructura)
DB_FILE = PROJECT_ROOT / "analytics.db"           # Archivo SQLite
CONSOLIDADO_TABLE = "consolidado"                 # Tabla base "real"
TERCEROS_CSV = PROJECT_ROOT / "data" / "terceros_sinteticos.csv"  # Sintéticos

print("DB_FILE:", DB_FILE)
print("SYNTH_CSV:", TERCEROS_CSV)


DB_FILE: C:\Users\pc\OneDrive\Documentos\Data project\tusdatos\analytics.db
SYNTH_CSV: C:\Users\pc\OneDrive\Documentos\Data project\tusdatos\data\terceros_sinteticos.csv


In [ ]:
# ============================================
# 3) (OPCIONAL) GENERAR BASE SINTÉTICA
# --------------------------------------------
# Este comando ejecuta tu generador de terceros sintéticos.
# Útil para que el notebook sea reproducible.
#
# Si YA tienes el CSV y no quieres regenerarlo cada vez,
# comenta esta celda (poniendo # al inicio de la línea) o bórrala.
# ============================================

# Crear base de terceros si no existe / si quieres regenerarla
# !python -m pipeline.matching.generate_terceros



OK: generado 10000 registros en data\terceros_sinteticos.csv
tipo_sujeto
PERSONA_NATURAL     7809
PERSONA_JURIDICA    2191
Name: count, dtype: int64
Docs nulos: 627


In [7]:
# ============================================
# 4) CARGA DE DATAFRAMES (SINTÉTICOS + CONSOLIDADO)
# --------------------------------------------
# df_terceros: dataset sintético con label "coincidencia" (1/0)
# df_cons: base consolidada (contra la que hacemos matching)
#
# Buenas prácticas:
# - dtype=str para evitar que documentos se vuelvan numéricos
# - fillna("") para no romper funciones por None/NaN
# ============================================

df_terceros = pd.read_csv(TERCEROS_CSV, dtype=str).fillna("")
print("terceros:", df_terceros.shape)

# Abrimos conexión a SQLite y leemos consolidado
conn = sqlite3.connect(DB_FILE)

# Importante: traemos solo columnas necesarias para el matching (ahorra memoria)
df_cons = pd.read_sql_query(
    f"""
    SELECT
        id_registro, fuente, tipo_sujeto, nombres, apellidos, fecha_nacimiento, nacionalidad,
        numero_documento, aliases
    FROM {CONSOLIDADO_TABLE}
    """,
    conn
).fillna("")

conn.close()
print("consolidado:", df_cons.shape)

# Vista rápida de terceros
df_terceros.head(2)


terceros: (10000, 10)
consolidado: (23613, 9)


,id_tercero,tipo_sujeto,nombres,apellidos,fecha_nacimiento,nacionalidad,numero_documento,tipo_documento,pais_residencia,coincidencia
0,T0000000,PERSONA_NATURAL,JUAN SARA,HERNANDEZ MARTINEZ,1964-03-24,VENEZOLANA,X2458591,PASAPORTE,ESPAÑA,0
1,T0000001,PERSONA_NATURAL,ANDRES LUISA,MARTINEZ TORRES,1988-01-18,PERUANA,U8038374,PASAPORTE,PERU,0


In [8]:
# Analisis rapido de terceros
df_terceros.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   id_tercero        10000 non-null  str  
 1   tipo_sujeto       10000 non-null  str  
 2   nombres           10000 non-null  str  
 3   apellidos         10000 non-null  str  
 4   fecha_nacimiento  10000 non-null  str  
 5   nacionalidad      10000 non-null  str  
 6   numero_documento  10000 non-null  str  
 7   tipo_documento    10000 non-null  str  
 8   pais_residencia   10000 non-null  str  
 9   coincidencia      10000 non-null  str  
dtypes: str(10)
memory usage: 1.6 MB


In [9]:
# ============================================
# 5) NORMALIZACIÓN + FEATURES BASE (nombre_full / doc_norm)
# --------------------------------------------
# Objetivo:
# - Convertir texto a una forma comparable entre datasets.
# - Evitar diferencias por: mayúsculas/minúsculas, tildes/símbolos, espacios dobles, etc.
#
# Outputs:
# - nombre_full: "NOMBRES APELLIDOS" normalizado (string)
# - doc_norm: documento normalizado (string)
# ============================================

def norm_text(s: str) -> str:
    # Manejo defensivo: NaN/None -> ""
    s = "" if s is None else str(s)

    # Estandarizamos a MAYÚSCULAS y sin espacios al inicio/fin
    s = s.upper().strip()

    # Dejamos solo letras, números y espacios (todo lo demás -> espacio)
    # Ej: "PÉREZ-LÓPEZ" -> "P REZ L PEZ" (si luego quieres preservar tildes, se puede mejorar)
    s = re.sub(r"[^A-Z0-9 ]+", " ", s)

    # Colapsamos espacios múltiples a uno solo
    s = re.sub(r"\s+", " ", s).strip()
    return s


def full_name(nombres: str, apellidos: str) -> str:
    # Normalizamos por separado y luego concatenamos
    n = norm_text(nombres)
    a = norm_text(apellidos)
    return (n + " " + a).strip()


# Creamos el nombre unificado en ambos dataframes
# (esto reduce complejidad: el modelo siempre compara nombre_full vs nombre_full)
df_terceros["nombre_full"] = df_terceros.apply(
    lambda r: full_name(r["nombres"], r["apellidos"]),
    axis=1
)
df_cons["nombre_full"] = df_cons.apply(
    lambda r: full_name(r["nombres"], r["apellidos"]),
    axis=1
)

# Normalizamos documento para poder usarlo en blocking (y en versiones futuras, como feature fuerte)
df_terceros["doc_norm"] = df_terceros["numero_documento"].apply(norm_text)
df_cons["doc_norm"] = df_cons["numero_documento"].apply(norm_text)

# Vista rápida de columnas claves para el experimento
df_terceros[["id_tercero", "nombre_full", "doc_norm", "coincidencia"]].head(5)


,id_tercero,nombre_full,doc_norm,coincidencia
0,T0000000,JUAN SARA HERNANDEZ MARTINEZ,X2458591,0
1,T0000001,ANDRES LUISA MARTINEZ TORRES,U8038374,0
2,T0000002,CARLOS JUAN RODRIGUEZ GARCIA,229494902,0
3,T0000003,ANA ANA RAMIREZ HERNANDEZ,D7350753,0
4,T0000004,DAVID CAMILA LOPEZ RAMIREZ,H5855124,0


In [10]:
# ============================================
# 6) CANDIDATE GENERATION (BLOCKING)
# --------------------------------------------
# Problema:
# Si comparas cada tercero contra TODO consolidado, el costo es:
#   O(N_terceros * N_consolidado)
# Eso explota rápido.
#
# Solución:
# "Blocking": generar un conjunto pequeño de candidatos probables.
# Aquí usamos dos estrategias simples:
#  1) Si hay doc_norm y existe en consolidado -> candidatos = todos con ese doc.
#  2) Si no hay doc -> candidatos por prefijo del nombre (primeros BLOCK_LEN chars).
#
# MAX_CANDIDATES:
# Evita bloques gigantes (ej: prefijos comunes como "MAR").
# ============================================

BLOCK_LEN = 3
MAX_CANDIDATES = 5000

# -------------------------
# 6.1) Índice por documento exacto
# doc_index: { "123456": [idx1, idx2, ...] }
# -------------------------
doc_index = {}
for idx, d in enumerate(df_cons["doc_norm"].values):
    if d:  # ignoramos vacíos
        # setdefault crea lista si no existe; append agrega el índice
        doc_index.setdefault(d, []).append(idx)

# -------------------------
# 6.2) Índice por bloque de nombre (prefijo)
# name_block_index: { "JUA": [idxs...], "CAR": [idxs...], ... }
# -------------------------
name_block_index = {}
for idx, n in enumerate(df_cons["nombre_full"].values):
    key = n[:BLOCK_LEN] if len(n) >= BLOCK_LEN else n
    name_block_index.setdefault(key, []).append(idx)

# -------------------------
# 6.3) Función para obtener candidatos de un tercero
# -------------------------
def get_candidate_indices(nombre_full: str, doc_norm: str) -> np.ndarray:
    # (1) Si hay doc y existe en el índice, usamos doc como blocking fuerte
    if doc_norm and doc_norm in doc_index:
        return np.array(doc_index[doc_norm], dtype=np.int32)

    # (2) Si no hay doc útil, hacemos blocking por prefijo de nombre
    key = nombre_full[:BLOCK_LEN] if len(nombre_full) >= BLOCK_LEN else nombre_full
    cands = name_block_index.get(key, [])

    # Recortamos por seguridad si el bloque es gigante
    if len(cands) > MAX_CANDIDATES:
        cands = cands[:MAX_CANDIDATES]

    return np.array(cands, dtype=np.int32)


In [12]:
# ============================================
# 7) MODELOS / SCORERS
# --------------------------------------------
# Cada función devuelve un score en [0,1] (idealmente):
# - Levenshtein / JaroWinkler: score continuo (0..1)
# - Fonético: score discreto (0 o 1)
# - Embeddings: cosine similarity (~0..1)
#
# NOTA:
# Para embeddings, el modelo se carga una sola vez (embedder),
# porque cargarlo dentro de la función sería MUY lento.
# ============================================

def score_lev(a: str, b: str) -> float:
    # Levenshtein normalizado: 1 = idéntico, 0 = completamente distinto
    if not a or not b:
        return 0.0
    return float(Levenshtein.normalized_similarity(a, b))


def score_jw(a: str, b: str) -> float:
    # Jaro-Winkler: favorece coincidencias con mismo prefijo (útil para nombres)
    if not a or not b:
        return 0.0
    return float(JaroWinkler.normalized_similarity(a, b))


def score_phon(a: str, b: str) -> float:
    # Double Metaphone: compara "sonido" aproximado.
    # Útil si "GONZALES" vs "GONZALEZ", etc.
    if not a or not b:
        return 0.0
    pa = doublemetaphone(a)[0]
    pb = doublemetaphone(b)[0]
    return 1.0 if (pa and pa == pb) else 0.0


# Cargamos el modelo de embeddings UNA sola vez
embedder = SentenceTransformer("all-MiniLM-L6-v2")


def score_emb(a: str, b: str) -> float:
    # Embeddings: convierte strings a vectores y mide similitud coseno
    if not a or not b:
        return 0.0

    # encode devuelve embeddings (vectores) para cada string
    emb = embedder.encode([a, b])

    # cosine_similarity devuelve matriz 1x1; extraemos el valor
    return float(cosine_similarity([emb[0]], [emb[1]])[0][0])


# Registro de modelos para poder iterar fácilmente
MODELS = {
    "levenshtein": score_lev,
    "jaro_winkler": score_jw,
    "phonetic": score_phon,
    #"embeddings": score_emb,
}


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 421.98it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# ============================================
# 7.1) Helpers para ALIAS y nombre "raw"
# --------------------------------------------
# - aliases en consolidado suelen venir como JSON string (lista)
# - necesitamos convertirlo a list[str] de forma segura
# - también construiremos nombres "tal como aparecen" para el reporte final
# ============================================
def json_load_list(x):
    """Devuelve una lista a partir de x (que puede ser list, JSON str, vacío)."""
    if x is None:
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        try:
            v = json.loads(s)
            return v if isinstance(v, list) else []
        except Exception:
            return []
    return []

def raw_full_name(nombres: str, apellidos: str) -> str:
    """Nombre sin normalizar (solo para mostrar en reportes)."""
    n = (nombres or "").strip()
    a = (apellidos or "").strip()
    return (n + " " + a).strip()


In [ ]:
# ============================================
# 8) RUNNER DE MATCHING (CONSISTENTE CON PRODUCTIVO)
# --------------------------------------------
# Para cada tercero (left):
#  1) Genera candidatos (blocking)
#  2) Aplica regla de negocio: si existe doc exacto entre candidatos -> ese match gana (prioridad)
#  3) Si no hay doc exacto, calcula score fuzzy contra:
#       - nombre principal del registro
#       - cada alias (si existe)
#     y toma el máximo (Top-1 global).
#
# Output por tercero:
# - best_score: mejor score (0..1)
# - best_doc_match: 1 si ganó por documento exacto
# - best_is_alias: 1 si el nombre ganador corresponde a un alias
# - best_id_right: id_registro del registro ganador en la lista
# - best_source: fuente del registro ganador
# - best_name_right: nombre/alias tal como aparece (raw) para el reporte
# - name_left: nombre normalizado del tercero (para depurar)
# - label: ground truth (coincidencia)
# ============================================

def run_top1(df_left: pd.DataFrame, model_name: str) -> pd.DataFrame:
    scorer = MODELS[model_name]
    out = []

    for _, r in df_left.iterrows():
        id_left = r["id_tercero"]
        nombre_left = r["nombre_full"]
        doc_left = r["doc_norm"]
        label = int(r["coincidencia"])

        cand_idx = get_candidate_indices(nombre_left, doc_left)

        # Sin candidatos: devolvemos vacío
        if cand_idx.size == 0:
            out.append({
                "id_left": id_left,
                "label": label,
                "best_score": 0.0,
                "best_doc_match": 0,
                "best_is_alias": 0,
                "best_id_right": None,
                "best_source": None,
                "best_name_right": "",
                "name_left": nombre_left,
            })
            continue

        cands = df_cons.iloc[cand_idx]

        # -----------------------------
        # (A) Prioridad: documento exacto
        # -----------------------------
        # Si el tercero trae documento y existe al menos un candidato con doc igual,
        # escogemos ese registro como match (regla de negocio de compliance).
        if doc_left:
            hit = cands[cands["doc_norm"] == doc_left]
            if len(hit) > 0:
                rr = hit.iloc[0]  # si hay varios, tomamos el primero (puedes refinar luego)
                out.append({
                    "id_left": id_left,
                    "label": label,
                    "best_score": 1.0,  # score "perfecto" porque ganó por doc
                    "best_doc_match": 1,
                    "best_is_alias": 0,
                    "best_id_right": rr["id_registro"],
                    "best_source": rr["fuente"],
                    "best_name_right": raw_full_name(rr.get("nombres",""), rr.get("apellidos","")),
                    "name_left": nombre_left,
                })
                continue

        # -----------------------------
        # (B) Fuzzy por nombre + aliases
        # -----------------------------
        best_score = -1.0
        best_doc_match = 0
        best_is_alias = 0
        best_id_right = None
        best_source = None
        best_name_right = ""

        for _, rr in cands.iterrows():
            # score contra nombre principal (normalizado)
            s_best = scorer(nombre_left, rr["nombre_full"])
            is_alias = 0
            name_raw = raw_full_name(rr.get("nombres",""), rr.get("apellidos",""))

            # score contra aliases (si existen)
            aliases = json_load_list(rr.get("aliases")) if "aliases" in rr else []
            for al in aliases:
                al_raw = str(al).strip()
                if not al_raw:
                    continue
                al_norm = norm_text(al_raw)  # normalizamos alias igual que nombres
                s_al = scorer(nombre_left, al_norm)
                if s_al > s_best:
                    s_best = s_al
                    is_alias = 1
                    name_raw = al_raw  # reportamos el alias tal cual

            if s_best > best_score:
                best_score = float(s_best)
                best_is_alias = int(is_alias)
                best_id_right = rr["id_registro"]
                best_source = rr["fuente"]
                best_name_right = name_raw

        out.append({
            "id_left": id_left,
            "label": label,
            "best_score": float(best_score if best_score >= 0 else 0.0),
            "best_doc_match": int(best_doc_match),  # aquí siempre 0 (si hubiera sido 1, entraba por rama doc)
            "best_is_alias": int(best_is_alias),
            "best_id_right": best_id_right,
            "best_source": best_source,
            "best_name_right": best_name_right,
            "name_left": nombre_left,
        })

    return pd.DataFrame(out)


# Ejemplo rápido: correr con una muestra chica para verificar que funciona
res_lev = run_top1(df_terceros.sample(200, random_state=42), "levenshtein")
res_lev.head()


,id_left,label,best_score,best_doc_match,name_left,best_name_right
0,T0006252,0,0.320000,0,ANA JORGE GOMEZ GOMEZ,ANALOG TECHNOLOGY LIMITED
1,T0004684,0,0.500000,0,ANDRES CAMILA MARTINEZ LOPEZ,ANDREA MARTINA LIMITED
2,T0001731,0,0.269231,0,LOPEZ LTDA,LOPEZ JOSE FRANCISCO LOPEZ
3,T0004742,0,0.366667,0,CAMILA CARLOS SANCHEZ MARTINEZ,CAMELIAS BAR S A DE C V
4,T0004521,0,0.302326,0,CAMILA JUAN LOPEZ MARTINEZ,CAMPOS TIRADO LINDA ELIZABETH CAMPOS TIRADO


In [ ]:
# ============================================
# 9) EVALUACIÓN (Precision / Recall / F1) POR UMBRAL
# --------------------------------------------
# Los modelos devuelven un score continuo. Para decidir "MATCH / NO MATCH"
# necesitamos un umbral:
#   pred = 1 si score >= threshold
#
# Métricas:
# - Precision: de los predichos como match, cuántos eran verdaderos
# - Recall: de los verdaderos match, cuántos detectamos
# - F1: balance entre precision y recall
#
# Nota de negocio (listas de sanciones):
# - Suele priorizarse recall alto (no perder matches reales),
#   pero sin disparar demasiados falsos positivos.
# ============================================

def eval_threshold(df_res: pd.DataFrame, threshold: float) -> dict:
    # y_true: etiqueta real (coincidencia)
    y_true = df_res["label"].values

    # y_pred: decisión por umbral
    y_pred = (df_res["best_score"].values >= threshold).astype(int)

    # Predicción por doc exacto (prioridad)
    pred_by_doc = (df_res["best_doc_match"].astype(int).values == 1).astype(int)

    # Regla final: doc manda
    y_pred = np.maximum(y_pred, pred_by_doc)

    # zero_division=0 evita warnings cuando no hay predicciones positivas
    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "positives": int(y_true.sum()),
        "predicted_positive": int(y_pred.sum()),
    }


def evaluate_model(df_left: pd.DataFrame, model_name: str, thresholds=(0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.93)): # Se puede ajustar el thresholds del modelo
    # 1) corremos matching top-1 
    df_res = run_top1(df_left, model_name)

    # 2) evaluamos a varios thresholds
    rows = [eval_threshold(df_res, t) for t in thresholds]

    # devolvemos el resumen (tabla) y el detalle por registro
    return pd.DataFrame(rows), df_res


# Corre en una muestra primero (para iterar rápido mientras ajustas)
df_sample = df_terceros.sample(1000, random_state=42)

# Correr modelo con la base de datos completa
df_sample = df_terceros

summary_lev, res_lev = evaluate_model(df_sample, "levenshtein")
summary_jw,  res_jw  = evaluate_model(df_sample, "jaro_winkler")
summary_ph,  res_ph  = evaluate_model(df_sample, "phonetic")
# summary_emb, res_emb = evaluate_model(df_sample, "embeddings") # Se deja comentado por el costo computacional de este modelo

# Muestra el resumen de un modelo (puedes cambiarlo por summary_jw, etc.)
summary_lev


,threshold,accuracy,precision,recall,f1,positives,predicted_positive
0,0.50,0.9182,0.492958,0.78750,0.606352,800,1278
1,0.55,0.9656,0.811475,0.74250,0.775457,800,732
2,0.60,0.9649,0.825835,0.71125,0.764271,800,689
3,0.65,0.9698,0.915000,0.68625,0.784286,800,600
4,0.70,0.9678,0.912069,0.66125,0.766667,800,580
5,0.75,0.9716,1.000000,0.64500,0.784195,800,516
6,0.80,0.9702,1.000000,0.62750,0.771121,800,502
7,0.85,0.9694,1.000000,0.61750,0.763524,800,494
8,0.90,0.9678,1.000000,0.59750,0.748044,800,478
9,0.93,0.9657,1.000000,0.57125,0.727128,800,457


In [ ]:
# ============================================
# 10) COMPARATIVO FINAL DE MODELOS
# --------------------------------------------
# Unimos los resúmenes de todos los modelos en una tabla para comparar.
# Ordenamos por threshold y luego por F1 para ver "quién gana" en cada umbral.
# ============================================

def add_model_col(df_summary: pd.DataFrame, model: str) -> pd.DataFrame:
    # Insertamos columna "model" al inicio para identificar el origen del resumen
    df2 = df_summary.copy()
    df2.insert(0, "model", model)
    return df2


comparativo = pd.concat([
    add_model_col(summary_lev, "levenshtein"),
    add_model_col(summary_jw,  "jaro_winkler"),
    add_model_col(summary_ph,  "phonetic"),
    # add_model_col(summary_emb, "embeddings"), # Se comenta porque el modelo tiene un alto costo computacional en comparación a los modelos clasicos
], ignore_index=True)

# Ordenamos para ver fácilmente el mejor F1 por umbral
comparativo.sort_values(["recall", "precision","f1"], ascending=[False, False, False])


,model,threshold,accuracy,precision,recall,f1,positives,predicted_positive
12,jaro_winkler,0.60,0.1028,0.079803,0.97000,0.147472,800,9724
11,jaro_winkler,0.55,0.0784,0.077849,0.97000,0.144131,800,9968
10,jaro_winkler,0.50,0.0776,0.077787,0.97000,0.144024,800,9976
13,jaro_winkler,0.65,0.1985,0.088138,0.96500,0.161523,800,8759
14,jaro_winkler,0.70,0.4444,0.118421,0.92250,0.209898,800,6232
15,jaro_winkler,0.75,0.4444,0.118421,0.92250,0.209898,800,6232
16,jaro_winkler,0.80,0.4684,0.122912,0.92000,0.216853,800,5988
17,jaro_winkler,0.85,0.8351,0.304468,0.82625,0.444968,800,2171
0,levenshtein,0.50,0.9182,0.492958,0.78750,0.606352,800,1278
1,levenshtein,0.55,0.9656,0.811475,0.74250,0.775457,800,732


## Conclusión – Selección de Modelo y Threshold

Después de evaluar los modelos de matching (Levenshtein, Jaro-Winkler y Fonético) bajo distintos thresholds, se tomó la decisión de seleccionar:

**Modelo:** Levenshtein  
**Threshold:** 0.50  

### Justificación Técnica

El criterio de selección priorizó un enfoque conservador desde la perspectiva de cumplimiento (compliance), donde el riesgo asociado a falsos negativos es considerablemente mayor que el de falsos positivos.

En un contexto de listas de sanciones o screening regulatorio:

- Un **falso negativo** implica no detectar una coincidencia real.
- Esto puede generar impactos reputacionales, regulatorios y financieros difíciles de cuantificar.
- En cambio, un **falso positivo** implica una revisión manual adicional, lo cual es operativamente manejable.

Con un threshold de 0.50, el modelo Levenshtein:

- Maximiza la detección de coincidencias reales (mayor recall comparado con thresholds más altos).
- Acepta un mayor número de falsos positivos como mecanismo de mitigación de riesgo.
- Mantiene un comportamiento estable y explicable desde el punto de vista técnico.

### Enfoque de Gestión de Riesgo

Se adopta deliberadamente una estrategia de:

> Preferir falsos positivos antes que falsos negativos.

Esto alinea el modelo con principios conservadores de cumplimiento normativo, donde la prioridad es evitar omitir coincidencias potencialmente críticas, incluso si esto incrementa el volumen de revisión manual.

### Próximos pasos (evolución futura)

- Evaluar ponderaciones multi-feature.
- Optimizar el threshold basado en curvas Precision-Recall y análisis operativo.

In [ ]:
df_cons

,tipo_sujeto,nombres,apellidos,fecha_nacimiento,nacionalidad,numero_documento,aliases,nombre_full,doc_norm
0,PERSONA_JURIDICA,AEROCARIBBEAN AIRLINES,,,[],,"[""AERO-CARIBBEAN""]",AEROCARIBBEAN AIRLINES,
1,PERSONA_JURIDICA,"ANGLO-CARIBBEAN CO., LTD.",,,[],,"[""AVIA IMPORT""]",ANGLO CARIBBEAN CO LTD,
2,PERSONA_JURIDICA,BANCO NACIONAL DE CUBA,,,[],,"[""BNC"", ""NATIONAL BANK OF CUBA""]",BANCO NACIONAL DE CUBA,
3,PERSONA_JURIDICA,BOUTIQUE LA MAISON,,,[],,[],BOUTIQUE LA MAISON,
4,PERSONA_JURIDICA,CASA DE CUBA,,,[],,[],CASA DE CUBA,
...,...,...,...,...,...,...,...,...,...
23608,PERSONA_NATURAL,"KARUSISI, RUKI",KARUSISI,,[],,[],KARUSISI RUKI KARUSISI,
23609,PERSONA_NATURAL,"NYAKARUNDI, VINCENT",NYAKARUNDI,,[],,[],NYAKARUNDI VINCENT NYAKARUNDI,
23610,PERSONA_NATURAL,"GASHUGI, STANISLAS",GASHUGI,,[],,[],GASHUGI STANISLAS GASHUGI,
23611,PERSONA_NATURAL,"MUGANGA, MUBARAKH",MUGANGA,,[],,"[""MK MUBARKH""]",MUGANGA MUBARAKH MUGANGA,


In [ ]:
# Generar reporte de alertas
# ==========================
# CONFIG (ajusta si quieres)
# ==========================
MODEL_NAME = "levenshtein"
THRESHOLD_MATCH = 0.50        # tu threshold seleccionado
THRESHOLD_REVIEW = 0.65       # zona gris: [match, review)
INCLUDE_ONLY_MATCHES = True   # True = solo los que pasan threshold o doc_exacto

# 1. Normalizar columnas esperadas en el resultado
df_res = res_lev.copy()

# Cambia aquí si tus nombres son distintos
COLMAP = {
    "id_left": "id_tercero",
    "best_id_right": "id_registro",
    "best_source": "fuente",
    "best_name_right": "nombre_lista",
    "best_score": "score_similitud",
    "best_doc_match": "doc_exacto",
    "best_is_alias": "is_alias",
}
for k, v in COLMAP.items():
    if k in df_res.columns and v not in df_res.columns:
        df_res = df_res.rename(columns={k: v})

# Fallbacks si no existen (para que no reviente)
if "doc_exacto" not in df_res.columns:
    df_res["doc_exacto"] = 0
if "is_alias" not in df_res.columns:
    df_res["is_alias"] = 0

# 2. Enriquecer con nombres del tercero (base sintética)
# nombre_tercero = nombres + apellidos (si aplica)
df_left = df_terceros[["id_tercero", "nombres", "apellidos", "numero_documento"]].copy()
df_left["nombre_tercero"] = (
    df_left["nombres"].fillna("").astype(str).str.strip() + " " +
    df_left["apellidos"].fillna("").astype(str).str.strip()
).str.strip()

# 3. Enriquecer con info de la lista (consolidado)
df_right = df_cons[["id_registro", "fuente", "nombres", "apellidos", "numero_documento"]].copy()
df_right["nombre_lista_full"] = (
    df_right["nombres"].fillna("").astype(str).str.strip() + " " +
    df_right["apellidos"].fillna("").astype(str).str.strip()
).str.strip()

# 4. Construir reporte (formato solicitado)
rep = (
    df_res.merge(df_left[["id_tercero", "nombre_tercero", "numero_documento"]], on="id_tercero", how="left", suffixes=("", "_terc"))
         .merge(df_right[["id_registro", "fuente", "nombre_lista_full", "numero_documento"]], on=["id_registro", "fuente"], how="left", suffixes=("", "_lista"))
)

# score 0-1
rep["score_similitud"] = rep["score_similitud"].astype(float)

# --------------------------
# Tipo match (regla simple)
# --------------------------
# doc exacto tiene prioridad
rep["tipo_match"] = np.where(
    rep["doc_exacto"].astype(int) == 1,
    "EXACTO_DOCUMENTO",
    np.where(
        rep["score_similitud"] >= 0.9999,
        "EXACTO_NOMBRE",
        np.where(
            rep["is_alias"].astype(int) == 1,
            "ALIAS",
            np.where(
                rep["score_similitud"] >= THRESHOLD_MATCH,
                "FUZZY_NOMBRE",
                "NO_MATCH"
            )
        )
    )
)

# --------------------------
# requiere_revision (zona gris)
# --------------------------
# Regla: solo si es match por score y cae en zona gris
rep["requiere_revision"] = False
mask_score_match = (rep["doc_exacto"].astype(int) == 0) & (rep["score_similitud"] >= THRESHOLD_MATCH)
mask_gray = mask_score_match & (rep["score_similitud"] < THRESHOLD_REVIEW)
rep.loc[mask_gray, "requiere_revision"] = True

# --------------------------
# Elegir nombre_lista visible:
# - si df_res ya trae el nombre exacto que matcheó, úsalo
# - si no, usa nombre_lista_full
# --------------------------
if "nombre_lista" not in rep.columns or rep["nombre_lista"].isna().all():
    rep["nombre_lista"] = rep["nombre_lista_full"]

# Filtrar solo matches si aplica
if INCLUDE_ONLY_MATCHES:
    rep = rep[(rep["tipo_match"] != "NO_MATCH")]

# Seleccionar columnas exactas del enunciado
alert_report = rep[[
    "id_tercero",
    "id_registro",
    "fuente",
    "tipo_match",
    "score_similitud",
    "nombre_tercero",
    "nombre_lista",
    "requiere_revision",
]].copy()

# Orden útil: primero lo más riesgoso/revisionable
alert_report = alert_report.sort_values(
    by=["requiere_revision", "score_similitud"],
    ascending=[False, False]
).reset_index(drop=True)

print("Alert report rows:", len(alert_report))
display(alert_report.head(20))

KeyError: 'id_registro'

In [ ]:
df_res

,id_tercero,label,score_similitud,doc_exacto,name_left,nombre_lista,is_alias
0,T0000000,0,0.392857,0,JUAN SARA HERNANDEZ MARTINEZ,JUAREZ CARTEL,0
1,T0000001,0,0.500000,0,ANDRES LUISA MARTINEZ TORRES,ANDREA MARTINA LIMITED,0
2,T0000002,0,0.409091,0,CARLOS JUAN RODRIGUEZ GARCIA,CARRENO ESCOBAR PEDRO MIGUEL CARRENO ESCOBAR,0
3,T0000003,0,0.357143,0,ANA ANA RAMIREZ HERNANDEZ,ANAYA MARTINEZ CESAR DANIEL ANAYA MARTINEZ,0
4,T0000004,0,0.423077,0,DAVID CAMILA LOPEZ RAMIREZ,DAVID YAFI DAVID,0
...,...,...,...,...,...,...,...
9995,T0009995,1,0.263158,1,ABDUL,ABDUL BASIR NOORZAI,0
9996,T0009996,1,0.173913,1,IMED,IMED BEN MEKKI ZARKAOUI,0
9997,T0009997,1,0.176471,1,KHALID,KHALID ABD AL RAHMAN HAMD AL FAWAZ,0
9998,T0009998,1,0.152174,1,IBRAHIM,IBRAHIM AWWAD IBRAHIM ALI AL BADRI AL SAMARRAI,0
